# Phase 7 - Evaluation

Project: ComplaintIQ

Work through this notebook incrementally. Validate each step before proceeding.

In [1]:
from pathlib import Path
PROJECT_ROOT = Path.home() / 'complaintiq'
print('Project root:', PROJECT_ROOT)
print('Exists:', PROJECT_ROOT.exists())

Project root: /home/sagemaker-user/complaintiq
Exists: True


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import time
import uuid
import sqlite3
import hashlib
import statistics
import traceback

import numpy as np
import pandas as pd

ROOT = Path("/home/sagemaker-user/complaintiq")
DATA = ROOT / "data"
PROC = DATA / "processed"
INDEX = ROOT / "index"
STATE = ROOT / "state"
SRC = ROOT / "src"
EVAL = ROOT / "eval"

EVAL.mkdir(parents=True, exist_ok=True)

GOLD_DIR = EVAL / "gold"
RESULT_DIR = EVAL / "results"

GOLD_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("Evaluation:", EVAL)
print("Golden sets:", GOLD_DIR)
print("Results:", RESULT_DIR)

assert ROOT.exists()
assert (PROC / "cfpb_holdout_june_2026.csv").exists()
assert (INDEX / "reg_vectors.npy").exists()
assert (INDEX / "case_vectors.npy").exists()

print("PASS — Evaluation directories and source artifacts verified.")

Project: /home/sagemaker-user/complaintiq
Evaluation: /home/sagemaker-user/complaintiq/eval
Golden sets: /home/sagemaker-user/complaintiq/eval/gold
Results: /home/sagemaker-user/complaintiq/eval/results
PASS — Evaluation directories and source artifacts verified.


In [23]:
from pathlib import Path
import json
import ast
import re

ROOT = Path("/home/sagemaker-user/complaintiq")
NOTEBOOK_DIR = ROOT / "notebooks"

SOURCE_NOTEBOOKS = {
    "Notebook3": NOTEBOOK_DIR / "03_retrieval.ipynb",
    "Notebook4": NOTEBOOK_DIR / "04_agents.ipynb",
    "Notebook5": NOTEBOOK_DIR / "05_graph.ipynb",
}

NOTEBOOK_CATALOG = {}

for notebook_name, notebook_path in SOURCE_NOTEBOOKS.items():

    assert notebook_path.exists(), (
        f"Notebook not found: {notebook_path}"
    )

    notebook = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    catalog = []

    for cell_number, cell in enumerate(notebook["cells"]):

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        if not source.strip():
            continue

        # Skip notebook magic commands for catalog parsing
        clean_source = "\n".join(
            line for line in source.splitlines()
            if not line.lstrip().startswith(("%", "!"))
        )

        try:
            tree = ast.parse(clean_source)
        except SyntaxError:
            continue

        definitions = []

        for node in tree.body:

            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                definitions.append(f"function: {node.name}")

            elif isinstance(node, ast.ClassDef):
                definitions.append(f"class: {node.name}")

            elif isinstance(node, (ast.Import, ast.ImportFrom)):
                definitions.append("import")

            elif isinstance(node, (ast.Assign, ast.AnnAssign)):
                targets = (
                    node.targets if isinstance(node, ast.Assign)
                    else [node.target]
                )

                for target in targets:
                    if isinstance(target, ast.Name):
                        definitions.append(f"variable: {target.id}")

        if definitions:
            catalog.append({
                "cell_number": cell_number,
                "definitions": definitions,
                "source": source
            })

    NOTEBOOK_CATALOG[notebook_name] = catalog

    print(f"\n{'=' * 65}")
    print(notebook_name)
    print(f"{'=' * 65}")

    for item in catalog:
        print(
            f"Cell {item['cell_number']:>3}: "
            + ", ".join(item["definitions"])
        )

print("\nPASS — Saved notebook definitions inventoried.")


Notebook3
Cell   1: import, variable: PROJECT_ROOT
Cell   2: import, import, variable: PROJECT_ROOT, import
Cell   3: import, import, import, import, variable: PROJECT, variable: DATA, variable: PROC, variable: INDEX, variable: REPORTS, variable: EVAL, variable: REGION, variable: BUCKET, variable: S3_INDEX
Cell   5: import, import, variable: PROJECT, variable: PROC, variable: INDEX_DIR, variable: S3_INDEX, variable: required, variable: doc_count, variable: pool_df, variable: holdout_df, variable: pool_count, variable: holdout_count, variable: pool_dates, variable: holdout_dates
Cell   7: import, import
Cell   8: import, variable: AWS_REGION, variable: session, variable: brt
Cell   9: variable: test_body, variable: response, variable: test_vector
Cell  11: variable: CACHE, function: cache_path, function: embed_one, function: embed
Cell  12: function: read_jsonl, variable: reg_chunks, variable: chunk_ids
Cell  14: variable: chunk_lengths
Cell  15: import, import, import, import, import,

In [3]:
import boto3

REGION = "us-east-1"
BUCKET = "sagemaker-complaintiq-456427397380"

MODELS = {
    "worker": "us.anthropic.claude-sonnet-4-6",
    "judge": "us.anthropic.claude-opus-4-8",
    "fast": "us.anthropic.claude-sonnet-4-6",
    "embed": "amazon.titan-embed-text-v2:0"
}

session = boto3.Session(region_name=REGION)

s3 = session.client("s3")
brt = session.client("bedrock-runtime")

# Verify S3 access
s3.head_bucket(Bucket=BUCKET)

# Verify Bedrock client configuration
assert brt.meta.region_name == REGION

print("Region:", REGION)
print("Bucket:", BUCKET)
print("Models:", MODELS)
print("PASS — AWS clients restored.")

Region: us-east-1
Bucket: sagemaker-complaintiq-456427397380
Models: {'worker': 'us.anthropic.claude-sonnet-4-6', 'judge': 'us.anthropic.claude-opus-4-8', 'fast': 'us.anthropic.claude-sonnet-4-6', 'embed': 'amazon.titan-embed-text-v2:0'}
PASS — AWS clients restored.


In [5]:

import duckdb

# --------------------------------------------------
# Phase 7 — Evaluation database validation
# --------------------------------------------------

DB_PATH = ROOT / "data" / "complaintiq.duckdb"

assert DB_PATH.exists(), f"Database not found: {DB_PATH}"

print("DuckDB:", DB_PATH)

# Close any existing read-only connection from this notebook
if "ro" in globals():
    try:
        ro.close()
    except Exception:
        pass

# Open database in read-only mode
ro = duckdb.connect(str(DB_PATH), read_only=True)

# Verify complaint view
view_check = ro.execute("""
    SELECT COUNT(*)
    FROM v_complaints
""").fetchone()[0]

print("v_complaints rows:", view_check)

assert view_check > 0, "v_complaints is empty"

print("PASS — Read-only evaluation database restored.")


DuckDB: /home/sagemaker-user/complaintiq/data/complaintiq.duckdb
v_complaints rows: 37923
PASS — Read-only evaluation database restored.


In [6]:
sys.path.insert(0, str(SRC))

import importlib
import llm

importlib.reload(llm)

assert callable(llm.chat)
assert callable(llm.get_usage_report)

print(json.dumps(
    llm.get_usage_report(),
    indent=2,
    default=str
))

print("PASS — Phase 4 LLM wrapper restored.")

{
  "ledger": {
    "calls": 0,
    "input_tokens": 0,
    "output_tokens": 0,
    "total_tokens": 0,
    "estimated_cost_usd": 0.0,
    "latency": {
      "calls": 0,
      "average_ms": 0,
      "p50_ms": 0,
      "p95_ms": 0
    },
    "models_used": []
  },
  "cache": {
    "cache_path": "/home/sagemaker-user/complaintiq/state/llm_cache.sqlite",
    "cached_responses": 9
  },
  "models_without_temperature": []
}
PASS — Phase 4 LLM wrapper restored.


In [9]:
class HybridIndex:

    def __init__(self, chunks, vectors):
        self.chunks = list(chunks)
        self.vectors = np.asarray(vectors, dtype="float32")

        if len(self.vectors) != len(self.chunks):
            raise ValueError("Vector/metadata count mismatch")

        if not np.isfinite(self.vectors).all():
            raise ValueError("Invalid vectors")

        self.faiss = faiss.IndexFlatIP(
            self.vectors.shape[1]
        )

        self.faiss.add(self.vectors)

        self.bm25 = BM25Okapi([
            tok(c.get("text", ""))
            for c in self.chunks
        ])

    def search(self, query, k=8, pool=40, rrf_k=60):

        if not query or not str(query).strip():
            raise ValueError("Query cannot be blank")

        pool = min(pool, len(self.chunks))

        qv = embed([query], workers=1)

        _, dense_ids = self.faiss.search(qv, pool)

        sparse_scores = self.bm25.get_scores(tok(query))

        sparse_ids = np.argsort(
            sparse_scores
        )[::-1][:pool]

        scores = defaultdict(float)

        for ranking in (dense_ids[0], sparse_ids):
            for rank, idx in enumerate(ranking):
                if int(idx) >= 0:
                    scores[int(idx)] += (
                        1.0 / (rrf_k + rank + 1)
                    )

        citation = re.search(
            r"\b(\d{1,2})\s*CFR\s*(\d{3,4}\.\d+)\b",
            query,
            re.I
        )

        if citation:
            target = (
                f"{citation.group(1)} CFR "
                f"{citation.group(2)}"
            ).lower()

            compact = (
                f"{citation.group(1)}cfr"
                f"{citation.group(2)}"
            ).lower()

            for idx, rec in enumerate(self.chunks):

                haystack = " ".join([
                    str(rec.get("chunk_id", "")),
                    str(rec.get("section", "")),
                    str(rec.get("text", ""))
                ]).lower()

                normalized = re.sub(
                    r"\s+", " ", haystack
                )

                compact_haystack = re.sub(
                    r"[^a-z0-9.]", "", haystack
                )

                if (
                    target in normalized
                    or compact in compact_haystack
                ):
                    scores[idx] += 1.0

        top = sorted(
            scores,
            key=scores.get,
            reverse=True
        )[:k]

        return [
            {
                **self.chunks[i],
                "rrf": round(float(scores[i]), 6)
            }
            for i in top
        ]



In [11]:
assert "HybridIndex" in globals()
assert callable(HybridIndex)

print("PASS — HybridIndex class restored.")

PASS — HybridIndex class restored.


In [13]:
import re

def tok(text):
    """
    Tokenize text for BM25 retrieval.
    Uses the same lowercase alphanumeric tokenization
    expected by the retrieval index.
    """
    return re.findall(r"\b\w+\b", str(text).lower())

print("PASS — tok() function defined.")
print("Test:", tok("12 CFR 1005.11 Electronic Fund Transfers"))

PASS — tok() function defined.
Test: ['12', 'cfr', '1005', '11', 'electronic', 'fund', 'transfers']


In [14]:
assert callable(tok)
assert callable(HybridIndex)

print("PASS — HybridIndex dependencies available.")

PASS — HybridIndex dependencies available.


In [15]:
import re
import faiss
from collections import defaultdict
from rank_bm25 import BM25Okapi

reg_chunks = [
    json.loads(line)
    for line in (INDEX / "reg_chunks.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

case_chunks = [
    json.loads(line)
    for line in (INDEX / "case_chunks.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

reg_vectors = np.load(INDEX / "reg_vectors.npy")
case_vectors = np.load(INDEX / "case_vectors.npy")

assert reg_vectors.shape == (1784, 1024)
assert case_vectors.shape == (3000, 1024)

# Re-execute the validated HybridIndex class and embed() implementation
# from Notebook 5's saved source cells before running this cell.

REG_INDEX = HybridIndex(reg_chunks, reg_vectors)
CASE_INDEX = HybridIndex(case_chunks, case_vectors)

assert REG_INDEX.faiss.ntotal == 1784
assert CASE_INDEX.faiss.ntotal == 3000

print("PASS — Phase 3 retrieval indexes reconstructed.")

PASS — Phase 3 retrieval indexes reconstructed.


In [17]:
def embed(texts, workers=3, dim=1024, desc="Embedding"):
    texts = list(texts)

    if not texts:
        return np.empty((0, dim), dtype="float32")

    with ThreadPoolExecutor(max_workers=workers) as executor:
        vectors = list(
            tqdm(
                executor.map(lambda t: embed_one(t, dim), texts),
                total=len(texts),
                desc=desc,
                dynamic_ncols=True
            )
        )

    return np.vstack(vectors).astype("float32")


In [20]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import numpy as np

print("PASS — Embedding dependencies imported.")

PASS — Embedding dependencies imported.


In [18]:
print("=== REGULATORY RETRIEVAL TEST ===")

results = REG_INDEX.search(
    "12 CFR 1005.11 error resolution requirements",
    k=5
)

for i, item in enumerate(results, 1):
    print(
        i,
        item.get("chunk_id"),
        str(item.get("text", ""))[:180]
    )

assert len(results) > 0

print("\nPASS — Regulatory retrieval operational.")

=== REGULATORY RETRIEVAL TEST ===


NameError: name 'ThreadPoolExecutor' is not defined

In [24]:
import json
from pathlib import Path

NOTEBOOKS = {
    "Notebook3": ROOT / "notebooks/03_retrieval.ipynb",
    "Notebook4": ROOT / "notebooks/04_agents.ipynb",
    "Notebook5": ROOT / "notebooks/05_graph.ipynb",
}

CELLS_TO_EXTRACT = {
    "Notebook3": [8, 11, 34],
    "Notebook4": [8, 10, 46, 48, 50, 52, 56, 59, 60, 61, 62, 63, 66, 67, 81, 104],
    "Notebook5": [14, 19, 22, 25, 26, 30, 34],
}

for notebook_name, cell_numbers in CELLS_TO_EXTRACT.items():

    nb_path = NOTEBOOKS[notebook_name]
    notebook = json.loads(nb_path.read_text(encoding="utf-8"))

    print("\n" + "=" * 90)
    print(notebook_name)
    print("=" * 90)

    for cell_number in cell_numbers:

        cell = notebook["cells"][cell_number]

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell.get("source", []))

        print(f"\n{'-' * 35} CELL {cell_number} {'-' * 35}")
        print(source)

print("\nPASS — Source extraction completed. No source cells were executed.")


Notebook3

----------------------------------- CELL 8 -----------------------------------
import boto3

AWS_REGION = "us-east-1"

session = boto3.Session(region_name=AWS_REGION)

brt = session.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION
)

print("AWS Account:", session.client("sts").get_caller_identity()["Account"])
print("AWS Region:", AWS_REGION)
print("Bedrock Runtime client initialized successfully.")

----------------------------------- CELL 11 -----------------------------------

CACHE = INDEX / "embed_cache"
CACHE.mkdir(parents=True, exist_ok=True)

def cache_path(text, dim=1024):
    key = hashlib.sha1(
        f"{dim}|{text}".encode("utf-8")
    ).hexdigest()
    return CACHE / f"{key}.npy"


def embed_one(text, dim=1024):
    text = str(text or "").strip() or " "
    path = cache_path(text, dim)

    if path.exists():
        return np.load(path).astype("float32")

    body = json.dumps({
        "inputText": text[:30000],
        "dimensions": dim